# Reproduce the project results

Upload the complete project ZIP below. This notebook extracts a separate copy, runs the shared model and tests, and offers the generated results for download. The calculation core uses Python's standard library; figures require matplotlib, available in a standard Colab runtime.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
assert len(uploaded) == 1, "Upload the complete project ZIP only."
archive_name = next(iter(uploaded))
extraction_dir = Path("project_run").resolve()
extraction_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_name) as archive:
    for member in archive.infolist():
        target = (extraction_dir / member.filename).resolve()
        if not target.is_relative_to(extraction_dir):
            raise ValueError("Archive contains an invalid extraction path.")
    archive.extractall(extraction_dir)

project_dir = next(
    path.parent for path in extraction_dir.glob("*/README.md")
)
assert (project_dir / "src/run_analysis.py").exists()
print("Project ready:", project_dir.name)

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "src.run_analysis", "--plots"],
    cwd=project_dir,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=project_dir,
    check=True,
)
print("Analysis and tests completed.")

In [ ]:
import shutil

results_archive = shutil.make_archive(
    "reproduced_project_results",
    "zip",
    root_dir=project_dir / "results",
)
files.download(results_archive)